In [ ]:
import kagglehub
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
csv_path = os.path.join(path, 'Q1_data.csv')
df = pd.read_csv(csv_path)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
df=df.drop( 'Order_ID',axis=1)
df.head()

In [ ]:
def check_missing_values(df):
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df)





In [ ]:
df['Weather']=df['Weather'].fillna(df['Weather'].mode()[0])
df['Traffic_Level']=df['Traffic_Level'].fillna(df['Traffic_Level'].mode()[0])
df['Time_of_Day']=df['Time_of_Day'].fillna(df['Time_of_Day'].mode()[0])
df['Courier_Experience_yrs']=df['Courier_Experience_yrs'].fillna(df['Courier_Experience_yrs'].mode()[0])
df['Delivery_Time']=df['Delivery_Time'].fillna(df['Delivery_Time'].median())

In [ ]:
check_missing_values(df)


In [ ]:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
categorical_cols = ['Weather','Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
df.head()

In [ ]:
cols= ['Weather','Traffic_Level', 'Time_of_Day', 'Vehicle_Type' , "	Distance_km", 'Preparation_Time_min','Courier_Experience_yrs', "Delivery_Time" ]
scl=StandardScaler()
df[cols] = scl.fit_transform(df[cols])

In [ ]:
check_target_distribution(df, "Delivery_Time")

In [ ]:
X = df.drop("Delivery_Time", axis=1).astype(float)
y = df["Delivery_Time"].astype(float)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error as sklearn_mse, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

print("Model trained!")

n_splits = 5 # K=5 Folds
loss=[]
pred=[]
# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate
    pred.append(y_pred)


    mse = sklearn_mse(y_test, y_pred)

    loss.append(mse)
average_losses = np.mean(loss, axis=0)

print(average_losses)

In [ ]:
feature_importance = pd.DataFrame({
    'feature': cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.hist(pred)
plt.show

In [ ]:
%pip install kagglehub catboost lightgbm tqdm -q
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
loss=[]

models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics


    # Store results
    loss.append(mse)
average_losses = np.mean(loss, axis=0)

print(average_losses ) #one for both sharing is good

